# 52 · Does a landmark model beat the free landmark ratio? An ablation ladder

Notebook 51 found two things.
- External landmarks repair within-segment stability.
- The zero-cost landmark ratio (LRS) gets most of the gain. Like every other arm, LCP fails P4c.

This notebook tests each component a model could add over LRS, one at a time, on the same external
landmark panels:

| Component | Arms compared |
|---|---|
| Per-gene NB likelihood with exposure | M0 against LRS |
| Non-monotone shapes | M1 against M0 |
| Shared, specimen-balanced paths | M1B against M1, under composition stress |
| Calibrated uncertainty | M1BC against LRS-U |
| Species registration through shared shapes | M1BCJ against M1BC |

**The fitter is gauge-anchored.**
- After every M-step, occupancy is made uniform.
- A coarse grid is followed by a fine one (50, then 200 points).
- This removes the flat direction that kept notebook 51's EM from converging.

**Every arm** also gets notebook 51's benchmark, the repaired P4c-dev, and the within-segment
residual transfer test (WSR) against a placebo coordinate.

**Decision recorded before any fit (coordinator).** P4c is a claim gate, not an adoption gate:
- no existing arm passes it;
- adoption uses (A) and (B) without P4c, plus the component ablations;
- if no component earns its margin, the method is "LRS with uncertainty".

Protocol: `SCRATCH/ws7/ledger_entry_phase3.md` and `plan_phase3.md`, written before any fit.
Kidney data only; no tissue coordinates anywhere. Cohort: two male control mice and two cortex
sections from one male human donor. Everything is descriptive.

## 1 · Folders and fixed settings

In [ ]:
import argparse
import json
import math
import os
import sys
from concurrent.futures import ProcessPoolExecutor
from hashlib import sha256
from multiprocessing import get_context
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream36 = results_root / 'pt_reconstruction_v2' / 'nb36'
external = data_root / 'external'
output_root = results_root / 'pt_reconstruction_v3'
output = output_root / 'nb52'
upstream51 = output_root / 'nb51'
output.mkdir(parents=True, exist_ok=True)
SAMPLES = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
source_paths = {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad'
                for names in SAMPLES.values() for n in names}
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'cross_species_nephron_scfates.h5ad',
            upstream13 / 'ortholog_map_used.csv', upstream36 / 'gene_sets_and_external_contrasts.csv',
            upstream36 / 'p4_heldout_prediction.csv',
            results_root / 'pt_reconstruction_v2' / 'nb35' / 'p2_absorption.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz',
            upstream51 / 'landmark_panel.csv', *source_paths.values()]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '52.landmark_ladder.1'  # Bump when a cached calculation changes.
EARLY_MARKERS, LATE_MARKERS = ('Slc5a2', 'Slc5a12', 'Gatm'), ('Slc22a7', 'Slc7a13', 'Cyp7b1')
N_FOLDS, MIN_DETECTION, ANCHOR_LOOSE = 3, .10, .5   # notebook 36
LANDMARK_THRESHOLD, LANDMARK_MIN_LEVEL = 1.0, 1.0
EQUAL_DEPTH_QUANTILE, SPLIT_EPSILON, SPLIT_SEED = .20, .5, 35   # notebook 36
MOUSE_SHAPE_POINTS, HUMAN_SHAPE_POINTS = (1 / 6, 1 / 2, 5 / 6), (1 / 3, 5 / 6)
INJECTION_SHARE_MOD, INJECTION_LOG_AMPLITUDE, OWN_INJECTION_SEED = 10, math.log(2), 357
BASIS_DF, P4C_DF = 6, 4
FIT_SETTINGS = dict(df=6, lam=10.0, coarse=50, fine=200, coarse_max=300, fine_max=500, tol_mean=1e-4,
                    tol_rel=1e-9, patience=5)
PLACEBO_MAX_CONTRAST, PLACEBO_SEED, STRESS_SHARE, STRESS_SEED = .2, 521, .7, 52
LRS_BOOT, LRS_BOOT_SEED, WSR_PERMUTATIONS, WSR_SEED, WSR_ALPHA = 200, 520, 500, 61, .01
CURVE_POINTS, CURVE_SHIFT, CONSERVED_MIN = 21, .25, .5
WORKERS = int(os.environ.get('PSEUDOSPACE_52_WORKERS', '36'))
print('Results folder:', output)

## 2 · Inputs: notebook 13's structures and the original counts (as notebook 36)

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from scipy.stats import spearmanr

import pseudospace.landmark_reconstruction as rec_module
from pseudospace.anchor_coordinate import (bulk_segment_profiles, census_segment_profiles, conserved_anchor_mask,
                                           positive_control_masks, pt_specific_mask, segment_contrast)
from pseudospace.coordinate_arms import heldout_prediction, registration_gap, within_segment_agreement
from pseudospace.coordinate_evaluation import (absorption_ratio, binomial_thin_to_depth, concordance,
                                               group_rank_correlations, log_normalize, poisson_count_split,
                                               scale_counts, sha256_folds)
from pseudospace.landmark_position import landmark_ratio_score, project_lcp, residual_spline_gain, scaled_gain
from pseudospace.landmark_reconstruction import (deviance_gain_repaired, design_effect_temperature,
                                                 fit_landmark_model, lrs_bootstrap_sd, project_landmark_model,
                                                 standardized_difference, wsr_correlations, wsr_statistic)
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.stage_cache import cached_payload, code_digest, digest, stage_is_fresh, stage_key
from pseudospace.within_segment_validation import fit_segment_covariate_oracle, predict_segment_covariate_oracle

saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r').obs.copy()
saved_neph = ad.read_h5ad(upstream13 / 'cross_species_nephron_scfates.h5ad', backed='r').obs.copy()
key_files = [p for p in sorted((upstream13 / 'stage_cache').glob('pass2_nephron_embedding__*.h5ad.key.json'))
             if "'logic': '13-nephron-pass2-v1'" in json.loads(p.read_text())['params']]
if len(key_files) != 1:
    raise FileNotFoundError(f'Expected one notebook-13 pass-2 cache, found {len(key_files)}; rerun notebook 13.')
neph = ad.read_h5ad(str(key_files[0])[:-len('.key.json')])
if not neph.obs_names.equals(saved_neph.index):
    raise ValueError('Notebook 13 pass-2 cache does not match its saved nephron structures.')
pt_rows = np.flatnonzero(neph.obs.coarse_class.astype(str).eq('PT').to_numpy())
if not neph.obs_names[pt_rows].equals(saved_pt.index):
    raise ValueError('PT rows differ from notebook 13 saved PT structures.')
rebuilt = rebuild_pt_expression(
    saved_neph[['sample', 'comparison_species', 'shared_pseudospace']].astype({'sample': str, 'comparison_species': str}),
    source_paths, pd.read_csv(upstream13 / 'ortholog_map_used.csv'), target_sum=1e4)
COUNTS = sparse.csr_matrix(rebuilt.layers['counts']).astype(np.int64)
genes_all = pd.Index(rebuilt.var_names)
measured_both = rebuilt.var.measured_in_both_inputs.to_numpy(bool)
cache_genes = pd.Index(neph.var_names)
cache_cols = genes_all.get_indexer(cache_genes)
if (cache_cols < 0).any() or abs(COUNTS[:, cache_cols] - sparse.csr_matrix(neph.layers['counts'])).max() != 0:
    raise ValueError('Rebuilt counts differ from notebook 13 cached counts.')
CONSTRUCTION_GENES = cache_genes[(neph.var.highly_variable_for_harmony & neph.var.highly_variable).to_numpy(bool)]
neph_obs = neph.obs.copy()
del neph

species_pt = saved_pt.comparison_species.astype(str).to_numpy()
specimen_pt = saved_pt['sample'].astype(str).to_numpy()
segment_pt = saved_pt.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
human_pt = species_pt == 'human'
SCF_SAVED = saved_pt.shared_pseudospace.to_numpy(float)
DPT_SAVED = saved_pt.dpt_pseudospace.to_numpy(float)
PT_COUNTS = COUNTS[pt_rows]
LIBRARY_PT = np.asarray(PT_COUNTS[:, measured_both].sum(axis=1)).ravel().astype(float)
if (LIBRARY_PT <= 0).any():
    raise ValueError('A PT structure has no counts in the shared panel.')
EVAL_ORIGINAL = log_normalize(PT_COUNTS, LIBRARY_PT)
detection = pd.DataFrame({sp: np.asarray((PT_COUNTS[species_pt == sp] > 0).mean(axis=0)).ravel()
                          for sp in SAMPLES}, index=genes_all)
print(f'{len(pt_rows):,} PT structures · {len(genes_all):,} genes · counts reproduce notebook 13')

## 3 · External references, notebook 36's evaluation sets and the landmark panels

- **Selection references** (landmarks only): mouse microdissection, female mouse snRNA, human snRNA
  donor half A.
- **Evaluation references** (metrics only): male mouse snRNA, including its adjacent-segment
  contrasts, and human half B.

A landmark has |S3 − early| ≥ 1 with the same sign in its species' selection references, level ≥ 1,
detection ≥ 10% in that species' PT and PT specificity in that species (notebook 36's rule, per
species). Notebook 36's conserved anchors and evaluation masks are rebuilt unchanged and checked.

In [ ]:
mouse_bulk = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
mouse_micro = bulk_segment_profiles(mouse_bulk.drop_duplicates('Gene_symbol').set_index('Gene_symbol'),
                                    '_PT{seg}_').reindex(genes_all)


def census(label, segment_map, early):
    counts = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    return census_segment_profiles(counts, meta.loc[counts.index], segment_map, early=early).reindex(genes_all)


mouse_sn = census('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                            'epithelial cell of proximal tubule segment 2': 'S2',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
human_sn = census('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, None)
human_donors = sorted(set(human_sn.columns.get_level_values('donor')))
HALF_A, HALF_B = human_donors[0::2], human_donors[1::2]
if HALF_A != ['21_015', 'HCA_51_ref', 'PRECISE-34'] or HALF_B != ['HCA_29_ref', 'HCA_55_ref', 'PRECISE-36']:
    raise ValueError(f'Human donor halves differ from the pre-registered split: {HALF_A} / {HALF_B}')


def micro_contrast(frame, late, early):
    return frame[late] - frame[early], frame[[late, early]].max(axis=1)


ext = {'select_mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
       'select_mouse_sn_female': segment_contrast(mouse_sn, sex='female'),
       'select_human_sn_halfA': segment_contrast(human_sn, donors=HALF_A),
       'mouse_sn_male': segment_contrast(mouse_sn, sex='male'),
       'mouse_sn_male_S2mS1': segment_contrast(mouse_sn, late='S2', early='S1', sex='male'),
       'mouse_sn_male_S3mS1': segment_contrast(mouse_sn, late='S3', early='S1', sex='male'),
       'mouse_sn_male_S3mS2': segment_contrast(mouse_sn, late='S3', early='S2', sex='male'),
       'mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
       'mouse_micro_S2mS1': micro_contrast(mouse_micro, 'S2', 'S1'),
       'mouse_micro_S3mS1': micro_contrast(mouse_micro, 'S3', 'S1'),
       'mouse_micro_S3mS2': micro_contrast(mouse_micro, 'S3', 'S2'),
       'human_sn_halfB': segment_contrast(human_sn, donors=HALF_B)}
contrast = pd.DataFrame({k: v[0] for k, v in ext.items()})
level = pd.DataFrame({k: v[1] for k, v in ext.items()})

classes = neph_obs.coarse_class.astype(str).to_numpy()
specimen_neph = neph_obs['sample'].astype(str).to_numpy()
library_neph = np.asarray(COUNTS[:, measured_both].sum(axis=1)).ravel().astype(float)
normalized = sparse.csr_matrix(COUNTS.multiply((1e4 / library_neph)[:, None]))
class_means = {}
for sp, names in SAMPLES.items():
    columns = {}
    for cls in sorted(set(classes)):
        per = [np.asarray(normalized[(classes == cls) & (specimen_neph == n)].mean(axis=0)).ravel()
               for n in names if ((classes == cls) & (specimen_neph == n)).sum() >= 20]
        columns[cls] = np.mean(per, axis=0)
    class_means[sp] = pd.DataFrame(columns, index=genes_all)
del normalized
eligible_anchor = (pd.Series(measured_both, index=genes_all) & detection.mouse.ge(MIN_DETECTION)
                   & detection.human.ge(MIN_DETECTION) & pt_specific_mask(class_means))
selection = ['select_mouse_micro', 'select_mouse_sn_female', 'select_human_sn_halfA']
A_LOOSE = conserved_anchor_mask(contrast[selection], level[selection], eligible_anchor, threshold=ANCHOR_LOOSE)
markers = pd.Series(genes_all.isin(EARLY_MARKERS + LATE_MARKERS), index=genes_all)
saved36 = pd.read_csv(upstream36 / 'gene_sets_and_external_contrasts.csv', index_col=0)
if not (saved36.anchor_loose_A166rule.astype(bool).reindex(genes_all).fillna(False) == A_LOOSE).all():
    raise ValueError("Notebook 36's conserved anchors do not reproduce.")


def evaluation_mask(sp, reference):
    return (pd.Series(measured_both, index=genes_all) & ~A_LOOSE & ~markers & detection[sp].ge(MIN_DETECTION)
            & level[reference].ge(1) & contrast[reference].notna())


control_eligible = (pd.Series(measured_both, index=genes_all) & ~A_LOOSE & ~markers & detection.mouse.ge(MIN_DETECTION)
                    & detection.human.ge(MIN_DETECTION) & level['mouse_sn_male'].ge(1))
D_OPPOSITE, D_FLAT = positive_control_masks(contrast['human_sn_halfB'], contrast['mouse_sn_male'], contrast['mouse_micro'],
                                            level['human_sn_halfB'], control_eligible)
FOLD = pd.Series(sha256_folds(genes_all, N_FOLDS), index=genes_all)
EVAL_GENES = genes_all[(evaluation_mask('mouse', 'mouse_sn_male') | evaluation_mask('human', 'human_sn_halfB')
                        | evaluation_mask('mouse', 'mouse_micro') | D_OPPOSITE | D_FLAT).to_numpy()]
EVAL_X = EVAL_ORIGINAL[:, genes_all.get_indexer(EVAL_GENES)]
fold_of_eval = FOLD.loc[EVAL_GENES].to_numpy()

# Landmark panels, one per species, from selection references only.
LANDMARK_RULES = {'mouse': ['select_mouse_micro', 'select_mouse_sn_female'], 'human': ['select_human_sn_halfA']}
panel_rows, LANDMARKS = [], {}
for sp, cols in LANDMARK_RULES.items():
    eligible = detection[sp].ge(MIN_DETECTION) & pt_specific_mask({sp: class_means[sp]})
    mask = conserved_anchor_mask(contrast[cols], level[cols], eligible, threshold=LANDMARK_THRESHOLD,
                                 min_level=LANDMARK_MIN_LEVEL)
    genes = list(genes_all[mask.to_numpy()])
    LANDMARKS[sp] = genes
    for g in genes:
        panel_rows.append({'species': sp, 'gene': g, 'sign': int(np.sign(contrast.loc[g, cols[0]])),
                           'fold': int(FOLD[g]), 'contrast_selection': float(contrast.loc[g, cols[0]])})
PANEL = pd.DataFrame(panel_rows)
PANEL.to_csv(output / 'landmark_panel.csv', index=False)
if not PANEL[['species', 'gene', 'sign']].equals(pd.read_csv(upstream51 / 'landmark_panel.csv')[['species', 'gene', 'sign']]):
    raise ValueError("The external landmark panels differ from notebook 51's.")
half_a = human_sn.loc[:, human_sn.columns.get_level_values('donor').isin(HALF_A)]
SHAPE_POINTS = {'mouse': np.array(MOUSE_SHAPE_POINTS), 'human': np.array(HUMAN_SHAPE_POINTS)}
SHAPE_SOURCE = {'mouse': mouse_micro[['S1', 'S2', 'S3']] * math.log(2),
                'human': pd.concat([half_a.xs('early', axis=1, level='segment').mean(axis=1),
                                    half_a.xs('S3', axis=1, level='segment').mean(axis=1)], axis=1) * math.log(2)}


def shape_values(sp, genes):
    """External segment-level log profile of each gene (selection references), genes x shape points."""
    values = SHAPE_SOURCE[sp].loc[list(genes)].to_numpy(float)
    if not np.isfinite(values).all():
        raise ValueError('Missing external profile for a panel gene.')
    return values


display(PANEL.groupby(['species', 'sign']).size().unstack().rename(columns={-1: 'early-high', 1: 'late-high'}))
print('Evaluation genes:', len(EVAL_GENES), '· landmarks also in the evaluation set:',
      {sp: int(pd.Index(g).isin(EVAL_GENES).sum()) for sp, g in LANDMARKS.items()})

## 4 · Comparators: SCF13 and DPT13 from notebook 36's cache, unchanged

The full fit must reproduce notebook 13's saved coordinates exactly before any refit is used.

In [ ]:
def load36(stage, spec=None):
    hits = []
    for path in upstream36.joinpath('stage_cache').glob(f'{stage}__*.npz'):
        params = json.loads(Path(str(path) + '.key.json').read_text())['params']
        if spec is None or f"'spec': '{spec}'" in params:
            hits.append(path)
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected one notebook-36 cache for {stage} {spec}, found {len(hits)}.')
    with np.load(hits[0], allow_pickle=False) as stored:
        return {k: stored[k] for k in stored.files}


SPECS = {'full': 'unperturbed', 'fold0': 'fold 0', 'fold1': 'fold 1', 'fold2': 'fold 2', 'ed': 'equal depth 352',
         'half_A': 'split A', 'half_B': 'split B'}
STAGES = {'full': 'arm_scf13_full', 'fold0': 'arm_scf13_fold0', 'fold1': 'arm_scf13_fold1', 'fold2': 'arm_scf13_fold2',
          'ed': 'arm_scf13_ed', 'half_A': 'arm_scf13_half_A', 'half_B': 'arm_scf13_half_B'}
fits36 = {key: load36(STAGES[key], SPECS[key]) for key in SPECS}
if np.abs(fits36['full']['scfates'] - SCF_SAVED).max() > 1e-9 or np.nanmax(np.abs(fits36['full']['dpt'] - DPT_SAVED)) > 1e-6:
    raise ValueError("Notebook 36's cached SCF13/DPT13 do not reproduce notebook 13.")
COORD = {'SCF13': {key: fit['scfates'] for key, fit in fits36.items()},
         'DPT13': {key: fit['dpt'] for key, fit in fits36.items()}}
COORD['SCF13']['loso'] = load36('loso_SCF13')['loso']
print('SCF13 and DPT13 refits loaded and reproduced.')

## 5 · Perturbed counts shared by every arm (notebook 36's seeds) and the LRS comparator

LRS is log((late + 0.5)/(early + 0.5)) over the species' landmark counts. A fold version drops that
fold's landmarks. For metrics that need a [0, 1] scale (splines, gaps), it is mapped to the
within-species pooled rank. That is monotone, so every Spearman metric is unchanged.

In [ ]:
def equal_depth(counts, seed):
    library = np.asarray(counts[:, measured_both].sum(axis=1)).ravel().astype(float)
    target = float(np.quantile(library[pt_rows], EQUAL_DEPTH_QUANTILE))
    return binomial_thin_to_depth(counts, library, target, np.random.default_rng(seed))


ED_352 = equal_depth(COUNTS, 352)
SPLIT_A, SPLIT_B = poisson_count_split(COUNTS, SPLIT_EPSILON, np.random.default_rng(SPLIT_SEED))


def hash_int(gene):
    return int(sha256(str(gene).encode()).hexdigest(), 16)


NONANCHOR_INJECTED = sorted([g for g in CONSTRUCTION_GENES if not A_LOOSE.get(g, False)
                             and g not in EARLY_MARKERS + LATE_MARKERS and hash_int(g) % INJECTION_SHARE_MOD == 0], key=hash_int)


def inject(counts, genes, s0, seed):
    """Human-only exp(±log 2 (2 s0 − 1)) on the listed genes; signs alternate in hash order (notebook 36)."""
    genes = sorted(genes, key=hash_int)
    signs = np.where(np.arange(len(genes)) % 2 == 0, 1.0, -1.0)
    f = INJECTION_LOG_AMPLITUDE * (2 * s0[human_pt] - 1)
    return scale_counts(counts, pt_rows[human_pt], genes_all.get_indexer(genes), f[:, None] * signs[None, :],
                        np.random.default_rng(seed)), genes


INJECTED_36, _ = inject(COUNTS, NONANCHOR_INJECTED, SCF_SAVED, 351)
VARIANTS = {'full': COUNTS, 'ed': ED_352, 'half_A': SPLIT_A, 'half_B': SPLIT_B}


def species_block(counts, sp, genes):
    rows = pt_rows[species_pt == sp]
    block = sparse.csr_matrix(counts[rows][:, genes_all.get_indexer(genes)])
    library = np.asarray(counts[rows][:, measured_both].sum(axis=1)).ravel().astype(float)
    return block, library


def fold_landmarks(sp, k=None):
    return [g for g in LANDMARKS[sp] if k is None or FOLD[g] != k]


def lrs(counts, k=None):
    out = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        genes = fold_landmarks(sp, k)
        sign = PANEL.set_index(['species', 'gene']).loc[sp].sign.loc[genes].to_numpy()
        block, _ = species_block(counts, sp, genes)
        late = np.asarray(block[:, sign > 0].sum(axis=1)).ravel()
        early = np.asarray(block[:, sign < 0].sum(axis=1)).ravel()
        score = landmark_ratio_score(late, early)
        out[species_pt == sp] = (pd.Series(score).rank().to_numpy() - .5) / len(score)
    return out


COORD['LRS'] = {'full': lrs(COUNTS), **{f'fold{k}': lrs(COUNTS, k) for k in range(N_FOLDS)},
                'ed': lrs(ED_352), 'half_A': lrs(SPLIT_A), 'half_B': lrs(SPLIT_B)}
print('LRS built for the full data, three folds, equal depth and both count-split halves.')


def lrs_raw(counts, sp, genes):
    """Raw landmark ratio for one species and gene list (no rank transform)."""
    sign = PANEL.set_index(['species', 'gene']).loc[sp].sign.loc[list(genes)].to_numpy()
    block, _ = species_block(counts, sp, genes)
    late = block[:, sign > 0].toarray()
    early = block[:, sign < 0].toarray()
    return landmark_ratio_score(late.sum(axis=1), early.sum(axis=1)), late, early

## 6 · The placebo panel and the comparator LCP (notebook 51's cached fits)

The placebo replaces each species' landmarks with the same number of non-zonal genes, each matched
to a landmark by the nearest log total count (seed 521). A non-zonal gene has |S3 − early| < 0.2 in
every selection reference. Detection and PT specificity are as for landmarks.

In [ ]:
PLACEBO = {}
rng_placebo = np.random.default_rng(PLACEBO_SEED)
for sp, cols in LANDMARK_RULES.items():
    eligible = (detection[sp].ge(MIN_DETECTION) & pt_specific_mask({sp: class_means[sp]})
                & contrast[cols].abs().lt(PLACEBO_MAX_CONTRAST).all(axis=1) & ~genes_all.isin(LANDMARKS[sp])
                & SHAPE_SOURCE[sp].notna().all(axis=1).reindex(genes_all).fillna(False).to_numpy())
    pool = list(rng_placebo.permutation(genes_all[eligible.to_numpy()]))
    totals = pd.Series(np.asarray(PT_COUNTS[species_pt == sp].sum(axis=0)).ravel(), index=genes_all)
    log_total = np.log1p(totals.loc[pool].to_numpy())
    free = np.ones(len(pool), bool)
    chosen = []
    for g in LANDMARKS[sp]:
        gap = np.where(free, np.abs(log_total - np.log1p(totals[g])), np.inf)
        best = int(np.argmin(gap))
        free[best] = False
        chosen.append(pool[best])
    PLACEBO[sp] = chosen
pd.DataFrame([{'species': sp, 'gene': g} for sp, gs in PLACEBO.items() for g in gs]).to_csv(output / 'placebo_panel.csv', index=False)
print('Placebo genes per species:', {sp: len(g) for sp, g in PLACEBO.items()})


def load51(stage):
    hits = list(upstream51.joinpath('stage_cache').glob(f'{stage}__*.npz'))
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected one notebook-51 cache for {stage}, found {len(hits)}.')
    with np.load(hits[0], allow_pickle=False) as stored:
        return {k: stored[k] for k in stored.files}


saved_lcp = pd.read_csv(output_root / 'lcp_coordinate.csv')
COORD['LCP'] = {}
for key in ('full', 'fold0', 'fold1', 'fold2', 'ed', 'half_A', 'half_B'):
    z = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        z[species_pt == sp] = load51(f'lcp_{sp}_{key}')['mean']
    COORD['LCP'][key] = z
if np.abs(COORD['LCP']['full'] - saved_lcp.position.to_numpy()).max() > 1e-12:
    raise ValueError("Notebook 51's cached LCP does not reproduce lcp_coordinate.csv.")
loso = np.full(len(pt_rows), np.nan)
for sp in SAMPLES:
    spec = specimen_pt[species_pt == sp]
    block, library = species_block(COUNTS, sp, LANDMARKS[sp])
    for name in SAMPLES[sp]:
        fit = load51(f'lcp_{sp}_loso_train_without_{name}')
        model = {'eta': fit['eta'], 'theta': fit['theta'], 'temperature': float(fit['temperature']), 'grid': fit['grid']}
        rows = spec == name
        loso[np.flatnonzero(species_pt == sp)[rows]] = project_lcp(model, block[rows].toarray(), library[rows])['mean']
COORD['LCP']['loso'] = loso
print('Comparator LCP loaded from notebook 51 and reproduced.')

## 7 · The ablation ladder: fits

**Arms.** All arms use the external landmark panels; PLACEBO uses the placebo panel.

| Arm | Configuration |
|---|---|
| M0 | linear paths, pooled |
| M1 | spline paths, pooled |
| M1B | spline paths, specimen-balanced |
| M1BC | M1B with a design-effect temperature |
| M1BCJ | M1BC fitted jointly across species, with shared shapes for conserved landmarks |
| PLACEBO | M1B on the placebo panel |

**Fits per arm and species:** full, three gene folds, equal depth, count-split halves A and B, and
LOSO for each specimen. Every arm except PLACEBO also gets the human own-landmark injection.

**Extra fits.**
- Composition stress for M1 and M1B: delete 70% of the S3 rows of Ctrl1A4 and of HUK1_MED1.
- Landmark gene halves: for the temperature (M1B configuration, T = 1), and at T for the
  calibration check.

**Runtime.** Every fit is cached. Fits run in parallel worker processes, and a progress log is
written as they finish.

In [ ]:
ARM_CFG = {'M0': dict(basis='linear', balanced=False), 'M1': dict(basis='spline', balanced=False),
           'M1B': dict(basis='spline', balanced=True), 'M1BC': dict(basis='spline', balanced=True),
           'M1BCJ': dict(basis='spline', balanced=True), 'PLACEBO': dict(basis='spline', balanced=True)}
MODEL_ARMS = ['M0', 'M1', 'M1B', 'M1BC', 'M1BCJ', 'PLACEBO']
cache_root = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(rec_module.__file__), project / 'pseudospace' / 'landmark_position.py'])
progress_log = output / 'progress.log'
CONSERVED = [g for g in LANDMARKS['mouse'] if g in set(LANDMARKS['human'])
             and PANEL.set_index(['species', 'gene']).sign.loc[('mouse', g)] == PANEL.set_index(['species', 'gene']).sign.loc[('human', g)]]
GENE_HALF = {g: hash_int(g) % 2 for sp in SAMPLES for g in LANDMARKS[sp] + PLACEBO[sp]}
STRESS_ROWS = {}
rng_stress = np.random.default_rng(STRESS_SEED)
for sp, target in (('mouse', 'Ctrl1A4'), ('human', 'HUK1_MED1')):
    spec, seg = specimen_pt[species_pt == sp], segment_pt[species_pt == sp]
    s3 = np.flatnonzero((spec == target) & (seg == 'S3'))
    drop = rng_stress.choice(s3, size=int(round(STRESS_SHARE * len(s3))), replace=False)
    keep = np.ones(len(spec), bool)
    keep[drop] = False
    STRESS_ROWS[sp] = keep


def make_block(counts, sp, genes, *, rows=None, temperature=1.0):
    blk, library = species_block(counts, sp, genes)
    spec = specimen_pt[species_pt == sp]
    if rows is not None:
        blk, library, spec = blk[rows], library[rows], spec[rows]
    return {'counts': blk.toarray().astype(np.int64), 'library': library, 'specimen': spec,
            'shape_points': SHAPE_POINTS[sp], 'shape_values': shape_values(sp, genes), 'temperature': float(temperature)}


def panel(arm, sp, k=None, half=None):
    genes = PLACEBO[sp] if arm == 'PLACEBO' else LANDMARKS[sp]
    genes = [g for g in genes if (k is None or FOLD[g] != k) and (half is None or GENE_HALF[g] == half)]
    return genes


def single_jobs(arm, sp, temperature=1.0, extra=True):
    jobs = {}
    for variant, counts in VARIANTS.items():
        jobs[f'{sp}_{variant}'] = [make_block(counts, sp, panel(arm, sp), temperature=temperature)]
    for k in range(N_FOLDS):
        jobs[f'{sp}_fold{k}'] = [make_block(COUNTS, sp, panel(arm, sp, k), temperature=temperature)]
    spec = specimen_pt[species_pt == sp]
    for name in SAMPLES[sp]:
        jobs[f'{sp}_loso_wo_{name}'] = [make_block(COUNTS, sp, panel(arm, sp), rows=spec != name, temperature=temperature)]
    return jobs


def joint_jobs(temps, counts_by_variant=None):
    counts_by_variant = counts_by_variant or VARIANTS
    jobs = {}
    for variant, counts in counts_by_variant.items():
        jobs[f'joint_{variant}'] = [make_block(counts, sp, LANDMARKS[sp], temperature=temps[sp]) for sp in SAMPLES]
    for k in range(N_FOLDS):
        jobs[f'joint_fold{k}'] = [make_block(COUNTS, sp, panel('M1BCJ', sp, k), temperature=temps[sp]) for sp in SAMPLES]
    for sp_out in SAMPLES:
        spec = specimen_pt[species_pt == sp_out]
        for name in SAMPLES[sp_out]:
            jobs[f'joint_loso_wo_{name}'] = [make_block(COUNTS, sp, LANDMARKS[sp], temperature=temps[sp],
                                                        rows=(spec != name) if sp == sp_out else None) for sp in SAMPLES]
    return jobs


def shared_pairs(blocks_genes):
    mouse_genes, human_genes = blocks_genes
    return [(mouse_genes.index(g), human_genes.index(g)) for g in CONSERVED if g in mouse_genes and g in human_genes]


def _run(job):
    from threadpoolctl import threadpool_limits
    with threadpool_limits(1):
        results, diag = fit_landmark_model(job['blocks'], **job['kwargs'])
    payload = {k: np.asarray(v) for k, v in diag.items()}
    for name, res in zip(job['names'], results):
        for key in ('mean', 'sd', 'q10', 'q90', 'map', 'entropy', 'eta', 'theta', 'grid'):
            payload[f'{name}__{key}'] = np.asarray(res[key])
        payload[f'{name}__temperature'] = np.asarray(res['temperature'])
    return payload


def run_jobs(jobs):
    """jobs: (arm, name) -> (blocks, gene lists for shared pairs or None). Cached; returns payloads."""
    from concurrent.futures import as_completed
    specs, missing = {}, []
    for (arm, name), (blocks, genes_lists) in jobs.items():
        kwargs = {**ARM_CFG[arm], **FIT_SETTINGS}
        if arm == 'M1BCJ':
            kwargs['shared'] = shared_pairs(genes_lists)
        names = list(SAMPLES) if len(blocks) == 2 else [name.split('_')[0]]
        params = dict(kwargs, temperatures=[b['temperature'] for b in blocks])
        inputs = {f'b{i}_{k}': b[k] for i, b in enumerate(blocks) for k in ('counts', 'library', 'specimen', 'shape_values')}
        key = stage_key(f'{arm}_{name}', params=params, inputs=inputs, code=code_digest(code))
        specs[(arm, name)] = (params, inputs, {'blocks': blocks, 'kwargs': kwargs, 'names': names})
        if not stage_is_fresh(cache_root / f'{arm}_{name}__{key}.npz', key):
            missing.append((arm, name))
    if missing:
        with ProcessPoolExecutor(max_workers=min(WORKERS, len(missing)), mp_context=get_context('fork')) as pool:
            futures = {pool.submit(_run, specs[m][2]): m for m in missing}
            for future in as_completed(futures):
                arm, name = futures[future]
                result = future.result()
                params, inputs, _ = specs[(arm, name)]
                # Cache each fit as soon as it finishes, so an interruption loses nothing.
                cached_payload(f'{arm}_{name}', lambda result=result: result, root=cache_root, params=params,
                               inputs=inputs, code=code, verbose=False)
                with open(progress_log, 'a') as handle:
                    handle.write(f'{(arm, name)} done\n')
    for (arm, name) in jobs:
        params, inputs, _ = specs[(arm, name)]
        FITS[arm][name] = cached_payload(f'{arm}_{name}', lambda: None, root=cache_root, params=params, inputs=inputs,
                                         code=code, verbose=False)


FITS = {arm: {} for arm in MODEL_ARMS}
# Wave 1: arms that need no temperature, plus the stress and gene-half fits.
jobs = {}
for arm in ('M0', 'M1', 'M1B', 'PLACEBO'):
    for sp in SAMPLES:
        jobs.update({(arm, name): (blocks, None) for name, blocks in single_jobs(arm, sp).items()})
        if arm in ('M1', 'M1B'):
            jobs[(arm, f'{sp}_stress')] = ([make_block(COUNTS, sp, panel(arm, sp), rows=STRESS_ROWS[sp])], None)
        if arm == 'M1B':
            for h in (0, 1):
                jobs[(arm, f'{sp}_genehalf{h}')] = ([make_block(COUNTS, sp, panel(arm, sp, half=h))], None)
run_jobs(jobs)

# Design-effect temperature per species from the two landmark gene halves (M1B configuration, T = 1).
TEMPERATURE, D_GENEHALF_T1 = {}, {}
for sp in SAMPLES:
    a, b = FITS['M1B'][f'{sp}_genehalf0'], FITS['M1B'][f'{sp}_genehalf1']
    TEMPERATURE[sp], D_GENEHALF_T1[sp] = design_effect_temperature(a[f'{sp}__mean'], a[f'{sp}__sd'],
                                                                   b[f'{sp}__mean'], b[f'{sp}__sd'])
print('Design-effect temperatures (frozen):', {k: round(v, 3) for k, v in TEMPERATURE.items()})

# Wave 2: calibrated arms.
jobs = {}
for sp in SAMPLES:
    jobs.update({('M1BC', name): (blocks, None) for name, blocks in single_jobs('M1BC', sp, temperature=TEMPERATURE[sp]).items()})
    for h in (0, 1):
        jobs[('M1BC', f'{sp}_genehalf{h}')] = ([make_block(COUNTS, sp, panel('M1BC', sp, half=h), temperature=TEMPERATURE[sp])], None)
for name, blocks in joint_jobs(TEMPERATURE).items():
    k = int(name[-1]) if name.startswith('joint_fold') else None
    jobs[('M1BCJ', name)] = (blocks, [panel('M1BCJ', sp, k) for sp in SAMPLES])
OWN_INJECTED_GENES = sorted([g for g in LANDMARKS['human'] if hash_int(g) % INJECTION_SHARE_MOD == 0], key=hash_int)
INJECTED = {}
for arm in ('M0', 'M1', 'M1B'):
    s0 = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        s0[species_pt == sp] = FITS[arm][f'{sp}_full'][f'{sp}__mean']
    INJECTED[arm], _ = inject(COUNTS, OWN_INJECTED_GENES, np.nan_to_num(s0), OWN_INJECTION_SEED)
    jobs[(arm, 'human_injection')] = ([make_block(INJECTED[arm], 'human', LANDMARKS['human'])], None)
run_jobs(jobs)


def full_position(arm, key='full'):
    z = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        fit = FITS[arm][f'joint_{key}'] if arm == 'M1BCJ' else FITS[arm][f'{sp}_{key}']
        z[species_pt == sp] = fit[f'{sp}__mean']
    return z


# Wave 3: own-landmark injection of the calibrated arms (human-only, 10% of human landmarks, s0 = own position).
jobs = {}
for arm in ('M1BC', 'M1BCJ'):
    INJECTED[arm], _ = inject(COUNTS, OWN_INJECTED_GENES, np.nan_to_num(full_position(arm)), OWN_INJECTION_SEED)
    if arm == 'M1BCJ':
        jobs[(arm, 'joint_injection')] = ([make_block(INJECTED[arm], sp, LANDMARKS[sp], temperature=TEMPERATURE[sp])
                                           for sp in SAMPLES], [LANDMARKS[sp] for sp in SAMPLES])
    else:
        jobs[(arm, 'human_injection')] = ([make_block(INJECTED[arm], 'human', LANDMARKS['human'],
                                                      temperature=TEMPERATURE['human'])], None)
run_jobs(jobs)
INJECTED['LRS'], _ = inject(COUNTS, OWN_INJECTED_GENES, COORD['LRS']['full'], OWN_INJECTION_SEED)

diag_rows = []
for arm, fits in FITS.items():
    for name, fit in fits.items():
        diag_rows.append({'arm': arm, 'fit': name, 'iterations coarse': int(fit['iterations_coarse']),
                          'iterations fine': int(fit['iterations_fine']), 'converged': bool(fit['converged'])})
FIT_DIAG = pd.DataFrame(diag_rows)
FIT_DIAG.to_csv(output / 'fit_diagnostics.csv', index=False)
CAP_SHARE = float((FIT_DIAG['iterations fine'] >= FIT_SETTINGS['fine_max']).mean())
print(f'{len(FIT_DIAG)} fits; share hitting the fine-stage cap: {CAP_SHARE:.3f} '
      f'(the convergence fix fails if > 0.10); not converged: {int((~FIT_DIAG.converged).sum())}')

## 8 · Coordinates per arm (posterior means), LOSO projections and SDs

In [ ]:
for arm in MODEL_ARMS:
    COORD[arm] = {key: full_position(arm, key) for key in ('full', 'fold0', 'fold1', 'fold2', 'ed', 'half_A', 'half_B')}
    loso = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        spec = specimen_pt[species_pt == sp]
        genes = panel(arm, sp)
        block, library = species_block(COUNTS, sp, genes)
        for name in SAMPLES[sp]:
            fit = FITS[arm][f'joint_loso_wo_{name}'] if arm == 'M1BCJ' else FITS[arm][f'{sp}_loso_wo_{name}']
            result = {'eta': fit[f'{sp}__eta'], 'theta': fit[f'{sp}__theta'], 'temperature': float(fit[f'{sp}__temperature']),
                      'grid': fit[f'{sp}__grid']}
            rows = spec == name
            loso[np.flatnonzero(species_pt == sp)[rows]] = project_landmark_model(result, block[rows].toarray(), library[rows])['mean']
    COORD[arm]['loso'] = loso


def arm_sd(arm, key):
    out = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        fit = FITS[arm][f'joint_{key}'] if arm == 'M1BCJ' else FITS[arm][f'{sp}_{key}']
        out[species_pt == sp] = fit[f'{sp}__sd']
    return out


ARMS = ['SCF13', 'DPT13', 'LRS', 'LCP'] + MODEL_ARMS
print('Arms:', ARMS)

## 9 · The benchmark of notebook 51, for every arm

- P1 and P1w are fold-protected and scored on evaluation references only. Microdissection P1w is
  reported for continuity and flagged as circular for the landmark arms.
- Also: gene-fold agreement, P5, the library correlation, LOSO, count split and registration gap.

In [ ]:
segment_group = np.where(human_pt, np.where(segment_pt == 'S3', 'S3', 'early'), segment_pt)
groups_within = np.char.add(np.char.add(specimen_pt.astype(str), '|'), segment_group.astype(str))
P1W_EVAL = {('mouse', 'S1'): 'mouse_sn_male_S2mS1', ('mouse', 'S2'): 'mouse_sn_male_S3mS1',
            ('mouse', 'S3'): 'mouse_sn_male_S3mS2', ('human', 'early'): 'human_sn_halfB', ('human', 'S3'): 'human_sn_halfB'}
P1W_MICRO = {('mouse', 'S1'): 'mouse_micro_S2mS1', ('mouse', 'S2'): 'mouse_micro_S3mS1', ('mouse', 'S3'): 'mouse_micro_S3mS2'}


def species_r(expression, z, genes):
    keep = np.isfinite(z)
    table = group_rank_correlations(expression[keep], z[keep], specimen_pt[keep])
    table.index = genes
    return pd.DataFrame({sp: table[names].mean(axis=1, skipna=False) for sp, names in SAMPLES.items()})


def within_r(expression, z, genes):
    keep = np.isfinite(z)
    table = group_rank_correlations(expression[keep], z[keep], groups_within[keep])
    table.index = genes
    return table


def protected(arm, function):
    parts = []
    for k in range(N_FOLDS):
        cols = fold_of_eval == k
        parts.append(function(EVAL_X[:, cols], COORD[arm][f'fold{k}'], EVAL_GENES[cols]))
    return pd.concat(parts).loc[EVAL_GENES]


rows, R = [], {}
for arm in ARMS:
    R[arm], RW = protected(arm, species_r), protected(arm, within_r)
    for sp, ref in (('mouse', 'mouse_sn_male'), ('human', 'human_sn_halfB'), ('mouse', 'mouse_micro')):
        mask = evaluation_mask(sp, ref).loc[EVAL_GENES].to_numpy()
        rows.append({'metric': 'P1', 'arm': arm, 'species': sp, 'segment': 'all', 'reference': ref,
                     'circular for landmark arms': ref == 'mouse_micro',
                     'value': concordance(R[arm].loc[mask, sp], contrast.loc[EVAL_GENES[mask], ref])[0]})
    for refs, tag in ((P1W_EVAL, False), (P1W_MICRO, True)):
        for (sp, seg), ref in refs.items():
            mask = evaluation_mask(sp, ref).loc[EVAL_GENES].to_numpy()
            for n in SAMPLES[sp]:
                rows.append({'metric': 'P1w', 'arm': arm, 'species': sp, 'segment': seg, 'specimen': n, 'reference': ref,
                             'circular for landmark arms': tag,
                             'value': concordance(RW.loc[mask, f'{n}|{seg}'], contrast.loc[EVAL_GENES[mask], ref])[0]})


def cell_rows(metric, arm, table, **extra):
    for rec in table.itertuples():
        sp = 'human' if rec.specimen in SAMPLES['human'] else 'mouse'
        rows.append({'metric': metric, 'arm': arm, 'species': sp, 'segment': rec.segment, 'specimen': rec.specimen,
                     'value': rec.rho, **extra})


log_library = np.log(LIBRARY_PT)
for arm in ARMS:
    c = COORD[arm]
    for a, b in ((0, 1), (0, 2), (1, 2)):
        cell_rows('gene-fold agreement', arm, within_segment_agreement(c[f'fold{a}'], c[f'fold{b}'], specimen_pt, segment_pt),
                  pair=f'{a}-{b}')
    cell_rows('P5 equal depth', arm, within_segment_agreement(c['ed'], c['full'], specimen_pt, segment_pt))
    cell_rows('count split A vs B', arm, within_segment_agreement(c['half_A'], c['half_B'], specimen_pt, segment_pt))
    cell_rows('position vs log library', arm, within_segment_agreement(c['full'], log_library, specimen_pt, segment_pt))
    if 'loso' in c:
        cell_rows('LOSO', arm, within_segment_agreement(c['loso'], c['full'], specimen_pt, segment_pt))
    for key in ('full', 'fold0', 'fold1', 'fold2', 'half_A', 'half_B'):
        gap, _ = registration_gap(c[key], species_pt, specimen_pt, segment_pt)
        for seg, value in gap.items():
            rows.append({'metric': 'registration gap (human − mouse median)', 'arm': arm, 'species': 'both',
                         'segment': seg, 'refit': key, 'value': float(value)})

### P2 · absorption of an injected human-only gradient

Each arm's own landmarks are injected, with s0 set to the arm's own full position. LRS is
recomputed. LCP's value is read from notebook 51, and SCF13 and DPT13 from notebook 35.

In [ ]:
def interaction(expression, z, genes):
    r = species_r(expression, z, pd.Index(genes))
    return (r.human - r.mouse).to_numpy()


p2_rows = []
cols_inj = genes_all.get_indexer(OWN_INJECTED_GENES)
for arm in ('LRS', 'M0', 'M1', 'M1B', 'M1BC', 'M1BCJ'):
    counts = INJECTED[arm]
    injected_eval = log_normalize(counts[pt_rows], np.asarray(counts[pt_rows][:, measured_both].sum(axis=1)).ravel().astype(float))
    z0 = COORD[arm]['full']
    if arm == 'LRS':
        z1 = lrs(counts)
    elif arm == 'M1BCJ':
        z1 = np.full(len(pt_rows), np.nan)
        for sp in SAMPLES:
            z1[species_pt == sp] = FITS[arm]['joint_injection'][f'{sp}__mean']
    else:
        z1 = z0.copy()
        z1[human_pt] = FITS[arm]['human_injection']['human__mean']
    original = interaction(EVAL_ORIGINAL[:, cols_inj], z0, OWN_INJECTED_GENES)
    frozen = interaction(injected_eval[:, cols_inj], z0, OWN_INJECTED_GENES)
    refitted = interaction(injected_eval[:, cols_inj], z1, OWN_INJECTED_GENES)
    moved = within_segment_agreement(z1, z0, specimen_pt, segment_pt)
    rec = {'arm': arm, **absorption_ratio(original, frozen, refitted, floor=.02),
           'human within-segment rho': float(moved[moved.specimen.isin(SAMPLES['human'])].rho.median())}
    rec['material absorption (notebook 35 flag)'] = bool(rec['median_ratio'] < .85 or rec['human within-segment rho'] < .90)
    p2_rows.append(rec)
p2_51 = pd.read_csv(upstream51 / 'p2_absorption.csv')
p2_51 = p2_51[p2_51.arm.isin(['LCP own landmarks', 'SCF13', 'DPT13'])].assign(arm=lambda d: d.arm.str.replace(' own landmarks', ''))
P2 = pd.concat([pd.DataFrame(p2_rows), p2_51], ignore_index=True)
P2.to_csv(output / 'p2_absorption.csv', index=False)
display(P2[['arm', 'median_ratio', 'slope', 'human within-segment rho', 'material absorption (notebook 35 flag)']])

### P4, P4c, P4c-dev (repaired) and WSR · held-out genes, cross-specimen within species

- **P4 and P4c** are unchanged from notebook 51.
- **P4c-dev** is repaired as protocol requires: genes with fewer than 10 training counts in a
  segment are excluded, and a ridge of 1 is used.
- **WSR** follows the protocol:
  - each specimen's residuals come from its own segment + coverage oracle;
  - per-gene, per-segment training splines are applied at test positions;
  - the statistic is the mean atanh(r) over all fold genes;
  - the null uses 500 within-segment permutations of the test coordinate, with seed 61 and the same
    permutations for every arm and fold.

Arms run in parallel processes.

In [ ]:
covariates = np.column_stack([log_library, np.log(neph_obs.n_spots.to_numpy(float)[pt_rows]),
                              np.log(np.asarray((COUNTS[pt_rows] > 0).sum(axis=1)).ravel())])
anatomy_code = pd.Series(segment_pt).map({'S1': 0, 'S2': 1, 'S3': 2}).to_numpy()
CELLS = []
RESID = {}
for sp, names in SAMPLES.items():
    genes_sp = evaluation_mask(sp, 'mouse_sn_male' if sp == 'mouse' else 'human_sn_halfB').loc[EVAL_GENES].to_numpy()
    for k in range(N_FOLDS):
        cols = genes_sp & (fold_of_eval == k)
        for n in names:
            m = specimen_pt == n
            y = EVAL_X[m][:, cols].toarray()
            own = fit_segment_covariate_oracle(y, covariates[m], anatomy_code[m], specimen_pt[m], degree=2)
            RESID[(sp, k, n)] = y - predict_segment_covariate_oracle(own, covariates[m], anatomy_code[m])
        count_cols = genes_all.get_indexer(EVAL_GENES[cols])
        for train, test in ((names[0], names[1]), (names[1], names[0])):
            tr, te = specimen_pt == train, specimen_pt == test
            ytr, yte = EVAL_X[tr][:, cols].toarray(), EVAL_X[te][:, cols].toarray()
            oracle = fit_segment_covariate_oracle(ytr, covariates[tr], anatomy_code[tr], specimen_pt[tr], degree=2)
            CELLS.append({'species': sp, 'fold': k, 'train': train, 'test': test, 'tr': tr, 'te': te, 'ytr': ytr, 'yte': yte,
                          'base_tr': predict_segment_covariate_oracle(oracle, covariates[tr], anatomy_code[tr]),
                          'base_te': predict_segment_covariate_oracle(oracle, covariates[te], anatomy_code[te]),
                          'ctr': PT_COUNTS[tr][:, count_cols].toarray(), 'cte': PT_COUNTS[te][:, count_cols].toarray()})
rng_wsr = np.random.default_rng(WSR_SEED)
PERMS = {}
for n in sorted(set(specimen_pt.astype(str))):
    for s in ('S1', 'S2', 'S3'):
        size = int(((specimen_pt == n) & (segment_pt == s)).sum())
        PERMS[(n, s)] = np.array([rng_wsr.permutation(size) for _ in range(WSR_PERMUTATIONS)])


def p4_arm(arm):
    out = []
    for cell in CELLS:
        z = COORD[arm][f"fold{cell['fold']}"]
        tr, te = cell['tr'], cell['te']
        p4 = heldout_prediction(cell['ytr'], z[tr], segment_pt[tr], cell['yte'], z[te], segment_pt[te], basis_df=BASIS_DF)
        spline = residual_spline_gain(cell['ytr'], z[tr], segment_pt[tr], cell['base_tr'], cell['yte'], z[te], segment_pt[te],
                                      cell['base_te'], basis_df=P4C_DF)
        dev = deviance_gain_repaired(cell['ctr'], LIBRARY_PT[tr], z[tr], segment_pt[tr], cell['cte'], LIBRARY_PT[te], z[te],
                                     segment_pt[te], basis_df=P4C_DF)
        for seg in ('all', 'S1', 'S2', 'S3'):
            r = np.ones(te.sum(), bool) if seg == 'all' else segment_pt[te] == seg
            var = np.maximum(p4['train_var'], 1e-12)
            out.append({'arm': arm, 'species': cell['species'], 'fold': cell['fold'], 'train': cell['train'], 'segment': seg,
                        'P4': (1 - np.sum(p4['coordinate_mse'] / var) / np.sum(p4['segment_mse'] / var)) if seg == 'all' else np.nan,
                        'P4c': scaled_gain(spline['base_mse'], spline['coordinate_mse'], spline['train_var'], r),
                        'P4c-dev (repaired)': float(1 - dev['coordinate_deviance'][r].sum() / dev['segment_deviance'][r].sum())})
    return out


def wsr_arm(arm):
    out = []
    for sp, names in SAMPLES.items():
        for train, test in ((names[0], names[1]), (names[1], names[0])):
            tr, te = specimen_pt == train, specimen_pt == test
            obs, null = {s: [] for s in ('S1', 'S2', 'S3')}, {s: [] for s in ('S1', 'S2', 'S3')}
            for k in range(N_FOLDS):
                z = COORD[arm][f'fold{k}']
                res = wsr_correlations(RESID[(sp, k, train)], z[tr], segment_pt[tr], RESID[(sp, k, test)], z[te], segment_pt[te],
                                       {s: PERMS[(test, s)] for s in ('S1', 'S2', 'S3')})
                for s, (o, nl) in res.items():
                    obs[s].append(o)
                    null[s].append(nl)
            for s in ('S1', 'S2', 'S3'):
                o, nl = np.concatenate(obs[s]), np.concatenate(null[s], axis=1)
                stat, null_stats = wsr_statistic(o), wsr_statistic(nl)
                out.append({'arm': arm, 'species': sp, 'segment': s, 'train': train, 'test': test, 'WSR': stat,
                            'null mean': float(np.mean(null_stats)), 'null max': float(np.max(null_stats)),
                            'p': float((1 + np.sum(null_stats >= stat)) / (1 + len(null_stats))), 'genes': int(len(o))})
    return out


def _limited(task):
    from threadpoolctl import threadpool_limits
    name, arm = task
    with threadpool_limits(1):
        return globals()[name](arm)


def _parallel(name, arms):
    with ProcessPoolExecutor(max_workers=min(WORKERS, len(arms)), mp_context=get_context('fork')) as pool:
        return [row for part in pool.map(_limited, [(name, a) for a in arms]) for row in part]


P4 = pd.DataFrame(_parallel('p4_arm', ARMS))
P4.to_csv(output / 'p4_p4c_heldout_prediction.csv', index=False)
WSR = pd.DataFrame(_parallel('wsr_arm', ARMS))
WSR.to_csv(output / 'wsr.csv', index=False)
saved_p4 = pd.read_csv(upstream36 / 'p4_heldout_prediction.csv')
check = P4[P4.segment.eq('all') & P4.arm.isin(['SCF13', 'DPT13'])].merge(saved_p4, on=['arm', 'species', 'fold', 'train'])
if len(check) != 24 or np.abs(check['P4'] - check['gain vs segment oracle']).max() > 1e-9:
    raise ValueError('SCF13/DPT13 P4 does not reproduce notebook 36.')
print('P4 for SCF13 and DPT13 reproduces notebook 36 (24 cells).')
display(P4.groupby(['species', 'segment', 'arm'])[['P4', 'P4c', 'P4c-dev (repaired)']].mean().unstack('arm').round(4))
display(WSR.pivot_table(index=['species', 'segment', 'test'], columns='arm', values='WSR').round(4))

## 10 · Calibration, composition stress and cross-species curves

**Calibration.** d is the difference of two estimates divided by √(sd₁² + sd₂²), for the read-split
pair (halves A and B) and the landmark gene-half pair. The targets are:
- SD(d) in [0.85, 1.15];
- a share of |d| ≤ 1.28 in [0.75, 0.85].

LRS-U uses bootstrap SDs (B = 200, seed 520). For read halves these are in raw LRS units. Gene-half
LRS values are not on a common raw scale, so they are compared on the within-species rank gauge,
with the delta-method SD sd_raw × density(LRS). This conversion was fixed before the fit.

In [ ]:
from scipy.stats import gaussian_kde

calib_rows, width_rows = [], []


def calib_cells(arm, pair, d, seg_of, sp):
    for s in ('S1', 'S2', 'S3'):
        m = seg_of == s
        sdd, share = float(np.std(d[m])), float(np.mean(np.abs(d[m]) <= 1.2815516))
        calib_rows.append({'arm': arm, 'pair': pair, 'species': sp, 'segment': s, 'SD of d': sdd, 'share |d| <= 1.28': share,
                           'calibration error': abs(sdd - 1),
                           'target met': bool(.85 <= sdd <= 1.15 and .75 <= share <= .85)})


LRS_U = {}
for sp in SAMPLES:
    seg_of = segment_pt[species_pt == sp]
    spec_of = specimen_pt[species_pt == sp]
    raw = {}
    for key, counts in (('full', COUNTS), ('half_A', SPLIT_A), ('half_B', SPLIT_B)):
        score, late, early = lrs_raw(counts, sp, LANDMARKS[sp])
        raw[key] = (score, lrs_bootstrap_sd(late, early, n_boot=LRS_BOOT, seed=LRS_BOOT_SEED))
    for h in (0, 1):
        genes = panel('LRS', sp, half=h)
        score, late, early = lrs_raw(COUNTS, sp, genes)
        sd_raw = lrs_bootstrap_sd(late, early, n_boot=LRS_BOOT, seed=LRS_BOOT_SEED + 1 + h)
        rank = (pd.Series(score).rank().to_numpy() - .5) / len(score)
        density = gaussian_kde(score)(score)
        raw[f'genehalf{h}'] = (rank, sd_raw * density)
    LRS_U[sp] = raw
    d_read = standardized_difference(raw['half_A'][0], raw['half_A'][1], raw['half_B'][0], raw['half_B'][1])
    d_gene = standardized_difference(raw['genehalf0'][0], raw['genehalf0'][1], raw['genehalf1'][0], raw['genehalf1'][1])
    calib_cells('LRS-U', 'read split', d_read, seg_of, sp)
    calib_cells('LRS-U', 'gene halves', d_gene, seg_of, sp)
    for s in ('S1', 'S2', 'S3'):
        for n in SAMPLES[sp]:
            m = (seg_of == s) & (spec_of == n)
            width_rows.append({'arm': 'LRS-U', 'species': sp, 'segment': s, 'specimen': n,
                               'relative width': float(np.median(raw['full'][1][m]) / np.std(raw['full'][0][m]))})
    for arm in ('M1B', 'M1BC'):
        a, b = FITS[arm][f'{sp}_half_A'], FITS[arm][f'{sp}_half_B']
        calib_cells(arm, 'read split', standardized_difference(a[f'{sp}__mean'], a[f'{sp}__sd'], b[f'{sp}__mean'], b[f'{sp}__sd']),
                    seg_of, sp)
        g0, g1 = FITS[arm][f'{sp}_genehalf0'], FITS[arm][f'{sp}_genehalf1']
        calib_cells(arm, 'gene halves', standardized_difference(g0[f'{sp}__mean'], g0[f'{sp}__sd'], g1[f'{sp}__mean'], g1[f'{sp}__sd']),
                    seg_of, sp)
        full = FITS[arm][f'{sp}_full']
        for s in ('S1', 'S2', 'S3'):
            for n in SAMPLES[sp]:
                m = (seg_of == s) & (spec_of == n)
                width_rows.append({'arm': arm, 'species': sp, 'segment': s, 'specimen': n,
                                   'relative width': float(np.median(full[f'{sp}__sd'][m]) / np.std(full[f'{sp}__mean'][m]))})
CALIB = pd.DataFrame(calib_rows)
WIDTH = pd.DataFrame(width_rows)
CALIB.to_csv(output / 'calibration.csv', index=False)
WIDTH.to_csv(output / 'relative_width.csv', index=False)
display(CALIB.round(3))

# Composition stress: untouched specimens' median |Δ| in segment units (unperturbed species boundaries).
def boundaries(z, seg):
    def cut(a, b):
        m = np.isin(seg, [a, b])
        cand = np.quantile(z[m], np.linspace(.02, .98, 97))
        return float(cand[int(np.argmax([np.mean((z[m] > t) == (seg[m] == b)) for t in cand]))])
    return cut('S1', 'S2'), cut('S2', 'S3')


def segment_units(z, b1, b2):
    return np.where(z < b1, z / b1, np.where(z < b2, 1 + (z - b1) / (b2 - b1), 2 + (z - b2) / (1 - b2)))


stress_rows = []
for sp, untouched in (('mouse', 'Ctrl1A2'), ('human', 'HUK1_COR1')):
    seg_of, spec_of = segment_pt[species_pt == sp], specimen_pt[species_pt == sp]
    keep = STRESS_ROWS[sp]
    for arm in ('LRS', 'M1', 'M1B'):
        if arm == 'LRS':
            score, _, _ = lrs_raw(COUNTS, sp, LANDMARKS[sp])
            before = (pd.Series(score).rank().to_numpy() - .5) / len(score)
            after_kept = (pd.Series(score[keep]).rank().to_numpy() - .5) / keep.sum()
        else:
            before = FITS[arm][f'{sp}_full'][f'{sp}__mean']
            after_kept = FITS[arm][f'{sp}_stress'][f'{sp}__mean']
        b1, b2 = boundaries(before, seg_of)
        u0 = segment_units(before[keep], b1, b2)
        u1 = segment_units(after_kept, b1, b2)
        sel = spec_of[keep] == untouched
        for s in ('S1', 'S2', 'S3'):
            m = sel & (seg_of[keep] == s)
            stress_rows.append({'arm': arm, 'species': sp, 'segment': s, 'median |Δ| (segment units)': float(np.median(np.abs(u1[m] - u0[m])))})
STRESS = pd.DataFrame(stress_rows)
STRESS.to_csv(output / 'composition_stress.csv', index=False)
display(STRESS.pivot_table(index=['species', 'segment'], columns='arm', values='median |Δ| (segment units)').round(4))

# Cross-species within-zone curves of conserved held-out genes (C-reg).
conserved_eval = (contrast[['mouse_sn_male', 'human_sn_halfB']].abs().ge(CONSERVED_MIN).all(axis=1)
                  & (np.sign(contrast.mouse_sn_male) == np.sign(contrast.human_sn_halfB))
                  & detection.mouse.ge(MIN_DETECTION) & detection.human.ge(MIN_DETECTION)
                  & ~genes_all.isin(LANDMARKS['mouse'] + LANDMARKS['human']))
CURVE_GENES = genes_all[conserved_eval.to_numpy()]
curve_x = EVAL_ORIGINAL[:, genes_all.get_indexer(CURVE_GENES)]
xs = np.linspace(0, 1, CURVE_POINTS)


def zone_curves(z, sp, s, shift=0.0):
    from pseudospace.stats_gam import gam_internal_knots, make_gam_design
    m = (species_pt == sp) & (segment_pt == s)
    lo, hi = np.quantile(z[m], [.01, .99])
    x = np.clip((z[m] - lo) / (hi - lo), 0, 1)
    knots = gam_internal_knots(x, basis_df=4)
    D = make_gam_design(x, knots)
    coef = np.linalg.lstsq(D, curve_x[m].toarray(), rcond=None)[0]
    return make_gam_design(np.clip(xs + shift, 0, 1), knots) @ coef


curve_rows = []
for arm in ('SCF13', 'DPT13', 'LRS', 'M1B', 'M1BC', 'M1BCJ'):
    z = COORD[arm]['full']
    for s in ('S1', 'S2', 'S3'):
        mouse_c = zone_curves(z, 'mouse', s)
        for shift in (0.0, -CURVE_SHIFT, CURVE_SHIFT):
            human_c = zone_curves(z, 'human', s, shift)
            rho = [spearmanr(mouse_c[:, g], human_c[:, g]).statistic for g in range(len(CURVE_GENES))]
            curve_rows.append({'arm': arm, 'segment': s, 'human shift': shift, 'median Spearman': float(np.nanmedian(rho))})
CURVES = pd.DataFrame(curve_rows)
CURVES.to_csv(output / 'cross_species_curves.csv', index=False)
print(len(CURVE_GENES), 'conserved held-out genes')
display(CURVES.pivot_table(index=['arm', 'segment'], columns='human shift', values='median Spearman').round(3))

## 11 · Summaries, component decisions, adoption and claim gates (as pre-registered)

In [ ]:
METRICS = pd.DataFrame(rows)
METRICS.to_csv(output / 'metrics_long.csv', index=False)
cells = METRICS[METRICS.metric.isin(['gene-fold agreement', 'P5 equal depth', 'count split A vs B', 'position vs log library', 'LOSO'])]
by_segment = cells.groupby(['metric', 'species', 'segment', 'arm']).value.median().unstack('arm')
by_segment.to_csv(output / 'summary_cells_by_segment.csv')
p1w = METRICS[METRICS.metric.eq('P1w')]
p1w.groupby(['circular for landmark arms', 'species', 'segment', 'arm']).value.mean().unstack('arm').to_csv(output / 'summary_p1w.csv')
P4.groupby(['species', 'segment', 'arm'])[['P4', 'P4c', 'P4c-dev (repaired)']].mean().unstack('arm').to_csv(output / 'summary_p4.csv')
gaps = METRICS[METRICS.metric.str.startswith('registration') & METRICS.refit.ne('full')]
display(by_segment.round(3))


def fold_summary(arm, sp, seg=None):
    t = METRICS[METRICS.metric.eq('gene-fold agreement') & METRICS.arm.eq(arm) & METRICS.species.eq(sp)]
    per = t.groupby('segment').value.median()
    return float(per.loc[seg]) if seg else float(per.mean())


def summary_row(arm):
    out = {}
    for sp in SAMPLES:
        out[f'gene-fold {sp}'] = fold_summary(arm, sp)
        out[f'P1w {sp}'] = float(p1w[p1w.arm.eq(arm) & p1w.species.eq(sp) & ~p1w['circular for landmark arms']].value.mean())
        ref = 'mouse_sn_male' if sp == 'mouse' else 'human_sn_halfB'
        out[f'P1 {sp}'] = float(METRICS[METRICS.metric.eq('P1') & METRICS.arm.eq(arm) & METRICS.reference.eq(ref)].value.iloc[0])
        cellsp = P4[P4.arm.eq(arm) & P4.species.eq(sp) & P4.segment.eq('all')]
        out[f'P4c mean {sp}'] = float(cellsp.P4c.mean())
        out[f'P4c > 0 cells {sp}'] = int((cellsp.P4c > 0).sum())
        out[f'P4 mean {sp}'] = float(cellsp.P4.mean())
    out['human S1 gene-fold'] = fold_summary(arm, 'human', 'S1')
    p5 = METRICS[METRICS.metric.eq('P5 equal depth') & METRICS.arm.eq(arm)]
    out['P5 overall'] = float(p5.value.median())
    out['P5 human S1'] = float(p5[p5.species.eq('human') & p5.segment.eq('S1')].value.mean())
    out['registration gap'] = float(gaps[gaps.arm.eq(arm)].value.abs().max())
    lo = METRICS[METRICS.metric.eq('LOSO') & METRICS.arm.eq(arm)]
    out['LOSO'] = float(lo.value.median()) if len(lo) else np.nan
    out['WSR mean'] = float(WSR[WSR.arm.eq(arm)].WSR.mean())
    return out


SUMMARY = pd.DataFrame({arm: summary_row(arm) for arm in ARMS})
SUMMARY.to_csv(output / 'summary_by_arm.csv')
display(SUMMARY.round(4))
S = SUMMARY

# Component decisions (plan §1, ledger protocol).
calib_err = CALIB.groupby('arm')['calibration error'].mean()
rel_width = WIDTH.groupby('arm')['relative width'].median()
stress = STRESS.groupby('arm')['median |Δ| (segment units)'].mean()
curve0 = CURVES[CURVES['human shift'].eq(0)].groupby('arm')['median Spearman'].mean()
headline = lambda a: [S.loc['gene-fold mouse', a], S.loc['P1w mouse', a], S.loc['human S1 gene-fold', a]]
COMPONENTS = {
    'C-lik (M0 vs LRS)': bool(S.loc['gene-fold mouse', 'M0'] >= S.loc['gene-fold mouse', 'LRS'] + .03
                              and S.loc['P1w mouse', 'M0'] >= S.loc['P1w mouse', 'LRS'] - .02
                              and S.loc['human S1 gene-fold', 'M0'] >= S.loc['human S1 gene-fold', 'LRS']),
    'C-shape (M1 vs M0)': bool((S.loc['gene-fold mouse', 'M1'] >= S.loc['gene-fold mouse', 'M0'] + .02
                                or S.loc['WSR mean', 'M1'] >= 1.10 * S.loc['WSR mean', 'M0'])
                               and all(x >= y - .02 for x, y in zip(headline('M1'), headline('M0')))),
    'C-bal (M1B vs M1)': bool(stress['M1B'] <= .5 * stress['M1'] and stress['M1B'] <= .5 * stress['LRS']),
    'C-unc (M1BC vs LRS-U)': bool(calib_err['M1BC'] <= calib_err['LRS-U'] and rel_width['M1BC'] <= rel_width['LRS-U']),
    'C-reg (M1BCJ vs M1BC)': bool(curve0['M1BCJ'] >= curve0['M1BC'] + .05)}
CALIBRATION_ABANDONED = bool((~CALIB[CALIB.arm.eq('M1BC')]['target met']).sum() > len(CALIB[CALIB.arm.eq('M1BC')]) / 2)
ladder = ['LRS-U', 'M0', 'M1', 'M1B', 'M1BC', 'M1BCJ']
adopted = 'LRS-U'
for component, arm in zip(COMPONENTS, ladder[1:]):
    if not COMPONENTS[component]:
        break
    adopted = arm
adopted_arm = 'LRS' if adopted == 'LRS-U' else adopted


def adoption(arm):
    a = S[arm]
    p2 = P2.set_index('arm')
    absorbed = bool(p2.loc[arm, 'material absorption (notebook 35 flag)']) if arm in p2.index and pd.notna(
        p2.loc[arm].get('material absorption (notebook 35 flag)', np.nan)) else False
    return {'(A) P1 >= SCF13 - 0.03, both species': bool(all(a[f'P1 {sp}'] >= S.loc[f'P1 {sp}', 'SCF13'] - .03 for sp in SAMPLES)),
            '(A) P5 >= 0.80 overall and >= 0.70 human S1': bool(a['P5 overall'] >= .80 and a['P5 human S1'] >= .70),
            '(A) registration gap <= 0.15': bool(a['registration gap'] <= .15),
            '(A) no material P2 absorption': not absorbed,
            '(B) P1w >= DPT13 + 0.05 mouse, >= DPT13 human': bool(a['P1w mouse'] >= S.loc['P1w mouse', 'DPT13'] + .05
                                                                  and a['P1w human'] >= S.loc['P1w human', 'DPT13']),
            '(B) gene-fold >= 0.70 mouse, >= 0.59 human': bool(a['gene-fold mouse'] >= .70 and a['gene-fold human'] >= .59),
            '(B) LOSO >= 0.85 (n/a counts as pass for the per-structure ratio)': bool(np.isnan(a['LOSO']) or a['LOSO'] >= .85)}


ADOPTION = {arm: adoption(arm) for arm in ARMS}
pd.DataFrame(ADOPTION).to_csv(output / 'adoption_criteria.csv')

# Claim gates (reported for every arm).
wsr = WSR.copy()
wsr['sig'] = wsr.p <= WSR_ALPHA
cellsig = wsr.groupby(['arm', 'species', 'segment']).sig.all()
cellstat = wsr.groupby(['arm', 'species', 'segment', 'test']).WSR.first()
claim_rows = []
for arm in ARMS:
    s = S[arm]
    p4c_claim = bool(s['P4c > 0 cells mouse'] >= 5 and s['P4c > 0 cells human'] >= 4 and all(
        s[f'P4c mean {sp}'] > max(S.loc[f'P4c mean {sp}', x] for x in ('SCF13', 'DPT13', 'LRS') if x != arm) for sp in SAMPLES))
    for sp in SAMPLES:
        for seg in ('S1', 'S2', 'S3'):
            passes = bool(cellsig.loc[(arm, sp, seg)])
            beats_placebo = bool(all(cellstat.loc[(arm, sp, seg, t)] > cellstat.loc[('PLACEBO', sp, seg, t)] for t in SAMPLES[sp]))
            placebo_fails = not bool(cellsig.loc[('PLACEBO', sp, seg)])
            claim_rows.append({'arm': arm, 'species': sp, 'segment': seg, 'WSR passes (p <= 0.01 both directions)': passes,
                               'beats placebo both directions': beats_placebo, 'placebo fails': placebo_fails,
                               'within-segment information claim': passes and beats_placebo and placebo_fails,
                               'held-out prediction claim (P4c)': p4c_claim})
CLAIMS = pd.DataFrame(claim_rows)
CLAIMS.to_csv(output / 'claim_gates.csv', index=False)
HUMAN_S1_UNRESOLVED = bool(S.loc['human S1 gene-fold', adopted_arm] < .50 or S.loc['P5 human S1', adopted_arm] < .70)
DECISION = {'components': COMPONENTS, 'calibration abandoned (targets fail in > half the cells)': CALIBRATION_ABANDONED,
            'convergence fix failed (> 10% of fits at the cap)': CAP_SHARE > .10, 'share of fits at the cap': CAP_SHARE,
            'adopted configuration': adopted, 'adoption criteria for the adopted configuration': ADOPTION[adopted_arm],
            'adopted passes (A) without the downstream gate and (B) without P4c': all(ADOPTION[adopted_arm].values()),
            'human S1 unresolved': HUMAN_S1_UNRESOLVED, 'temperatures': TEMPERATURE,
            'conserved landmarks shared in M1BCJ': len(CONSERVED)}
print(json.dumps(DECISION, indent=1, default=str))
display(CLAIMS.pivot_table(index=['species', 'segment'], columns='arm', values='within-segment information claim'))

## 12 · Coordinate files for workstream B and the record

Every arm is written in `primary_pt_coordinate.csv` format. Model arms add posterior columns, and
LRS-U adds its bootstrap SD. Workstream B's notebooks run on the surviving arms and on the placebo.

In [ ]:
coord_dir = output_root / 'coordinates'
coord_dir.mkdir(exist_ok=True)
for arm in ['LRS'] + MODEL_ARMS:
    frame = pd.DataFrame({'structure_id': saved_pt.index, 'specimen': specimen_pt, 'species': species_pt,
                          'segment_class': saved_pt.segment_class.astype(str).to_numpy(), 'position': COORD[arm]['full'],
                          'coordinate_method': arm, 'method_description': f'notebook 52 arm {arm}'})
    if arm == 'LRS':
        sd = np.full(len(pt_rows), np.nan)
        for sp in SAMPLES:
            sd[species_pt == sp] = LRS_U[sp]['full'][1]
        frame['bootstrap_sd_raw'] = sd
    else:
        frame['posterior_sd'] = arm_sd(arm, 'full')
    frame.to_csv(coord_dir / f'{arm.lower()}.csv', index=False)
with open(output / 'summary.json', 'w') as handle:
    json.dump({'logic_version': NOTEBOOK_LOGIC_VERSION, 'decision': DECISION, 'summary_by_arm': S.round(6).to_dict(),
               'calibration_error': calib_err.to_dict(), 'relative_width': rel_width.to_dict(), 'stress': stress.to_dict(),
               'curves': curve0.to_dict(), 'n_fits': len(FIT_DIAG)}, handle, indent=1, default=str)
print('Saved coordinates to', coord_dir)